<a href="https://colab.research.google.com/github/sethkipsangmutuba/Data-Visualization/blob/main/Session_A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sorting Arrays
##Seth Kipsang

### Fast Sorting in NumPy: np.sort and np.argsort

### Partial Sorts: Partitioning

### Example: k-Nearest Neighbors

## Sorting Arrays

Focuses on methods for ordering values in NumPy arrays.

Basic sorting algorithms (e.g., selection sort) are simple but inefficient, especially for large datasets (O(N²) complexity).

Some naive methods (e.g., random shuffling) are extremely inefficient and impractical.

Efficient sorting relies on optimized built-in algorithms in Python and NumPy.

---

## Key idea

Use optimized sorting functions rather than manual implementations for performance.

##8.1) Fast Sorting in NumPy: np.sort and np.argsort

NumPy provides efficient sorting functions, typically using an O(N log N) algorithm (quicksort by default).

`np.sort` returns a sorted copy of the array without modifying the original.

`array.sort()` performs sorting in-place, changing the original array.

`np.argsort` returns the indices that would sort the array.

These indices can be used with fancy indexing to reconstruct the sorted array.

---

## Key idea

NumPy sorting is efficient, flexible, and supports both value-based and index-based sorting.

In [76]:
import pandas as pd
from sklearn.datasets import load_iris

# Load dataset
iris = load_iris()

# Convert to DataFrame
iris_df = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

# Optional: add target column
iris_df['target'] = iris.target

iris_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [77]:
import numpy as np

# Iris dataset

X = iris_df.iloc[:, :4].to_numpy()

# Focus on one feature for clarity
sepal_length = X[:, 0]

# 1. np.sort → returns sorted copy

sorted_values = np.sort(sepal_length)

print("Sorted values (first 10):", sorted_values[:10])

# Original remains unchanged
print("Original (first 10):", sepal_length[:10])


Sorted values (first 10): [4.3 4.4 4.4 4.4 4.5 4.6 4.6 4.6 4.6 4.7]
Original (first 10): [5.1 4.9 4.7 4.6 5.  5.4 4.6 5.  4.4 4.9]


In [78]:
# 2. In-place sorting

x_copy = sepal_length.copy()
x_copy.sort()

print("\nIn-place sorted (first 10):", x_copy[:10])



In-place sorted (first 10): [4.3 4.4 4.4 4.4 4.5 4.6 4.6 4.6 4.6 4.7]


In [79]:
# 3. np.argsort → indices that sort the array

sort_idx = np.argsort(sepal_length)

print("\nArgsort indices (first 10):", sort_idx[:10])

# Reconstruct sorted array using indices
sorted_via_idx = sepal_length[sort_idx]

print("Sorted via argsort (first 10):", sorted_via_idx[:10])



Argsort indices (first 10): [13  8 42 38 41 22  3  6 47  2]
Sorted via argsort (first 10): [4.3 4.4 4.4 4.4 4.5 4.6 4.6 4.6 4.6 4.7]


In [80]:
# 4. Sorting full dataset by one column

sorted_rows = X[sort_idx]

print("\nDataset sorted by sepal length (first 5 rows):")
print(sorted_rows[:5])


Dataset sorted by sepal length (first 5 rows):
[[4.3 3.  1.1 0.1]
 [4.4 2.9 1.4 0.2]
 [4.4 3.2 1.3 0.2]
 [4.4 3.  1.3 0.2]
 [4.5 2.3 1.3 0.3]]


##8.2) Sorting along rows or columns

NumPy allows sorting along a specific axis in multidimensional arrays.

Using `axis=0` sorts each column independently.

Using `axis=1` sorts each row independently.

Each row or column is treated separately, so relationships across them are not preserved.

---

## Key idea

Axis-based sorting organizes data within rows or columns, not across the entire array.

In [81]:
import numpy as np

# Iris dataset

X = iris_df.iloc[:, :4].to_numpy()

# 1. Sort along columns (axis=0)

col_sorted = np.sort(X, axis=0)

print("Column-wise sorted (first 5 rows):")
print(col_sorted[:5])


Column-wise sorted (first 5 rows):
[[4.3 2.  1.  0.1]
 [4.4 2.2 1.1 0.1]
 [4.4 2.2 1.2 0.1]
 [4.4 2.2 1.2 0.1]
 [4.5 2.3 1.3 0.1]]


In [82]:
# 2. Sort along rows (axis=1)

row_sorted = np.sort(X, axis=1)

print("\nRow-wise sorted (first 5 rows):")
print(row_sorted[:5])


Row-wise sorted (first 5 rows):
[[0.2 1.4 3.5 5.1]
 [0.2 1.4 3.  4.9]
 [0.2 1.3 3.2 4.7]
 [0.2 1.5 3.1 4.6]
 [0.2 1.4 3.6 5. ]]


##8.3) Partial Sorts: Partitioning

Sometimes only the K smallest values are needed rather than a full sort.

`np.partition` places the K smallest elements on the left side, with the rest on the right in arbitrary order.

Elements within each partition are not sorted.

Partitioning can also be applied along a specific axis in multidimensional arrays.

`np.argpartition` returns the indices corresponding to the partition.

---

## Key idea

Partitioning is faster than full sorting when only partial order is required.

In [83]:
import numpy as np

# -----------------------------
# Iris dataset
# -----------------------------
X = iris_df.iloc[:, :4].to_numpy()

# Focus on one feature
sepal_length = X[:, 0]

# 1. np.partition → K smallest values

k = 5

partitioned = np.partition(sepal_length, k)

print("First 10 after partition:", partitioned[:10])
print("K smallest values (unsorted):", partitioned[:k])

First 10 after partition: [4.3 4.4 4.4 4.4 4.5 4.6 4.6 4.6 4.6 4.7]
K smallest values (unsorted): [4.3 4.4 4.4 4.4 4.5]


Using np.argpartition (Index-based)

In [84]:
idx = np.argpartition(sepal_length, k)

smallest_k = sepal_length[idx[:k]]

print("\nIndices of K smallest:", idx[:k])
print("K smallest via indices:", smallest_k)


Indices of K smallest: [13  8 42 38 41]
K smallest via indices: [4.3 4.4 4.4 4.4 4.5]


Apply to Full Dataset

In [85]:
row_idx = np.argpartition(sepal_length, k)

top_rows = X[row_idx[:k]]

print("\nRows with smallest sepal length:")
print(top_rows)


Rows with smallest sepal length:
[[4.3 3.  1.1 0.1]
 [4.4 2.9 1.4 0.2]
 [4.4 3.2 1.3 0.2]
 [4.4 3.  1.3 0.2]
 [4.5 2.3 1.3 0.3]]


Axis-Based Partitioning

In [86]:
# Partition each column independently
col_partitioned = np.partition(X, k, axis=0)

print("\nColumn-wise partition (first 5 rows):")
print(col_partitioned[:5])


Column-wise partition (first 5 rows):
[[4.3 2.  1.  0.1]
 [4.4 2.2 1.1 0.1]
 [4.4 2.2 1.2 0.1]
 [4.4 2.2 1.2 0.1]
 [4.5 2.3 1.3 0.1]]


## Example: k-Nearest Neighbors

Uses `argsort` along axes to find nearest neighbors.

Compute pairwise squared distances using broadcasting and aggregation.

Distance matrix has zeros along the diagonal.

Sorting each row gives indices of nearest neighbors.

First column corresponds to each point itself.

Full sorting may be unnecessary for nearest neighbors.

Use `argpartition` to get k nearest neighbors more efficiently.

Visualization connects each point to its nearest neighbors.

Neighbor relationships are not necessarily symmetric.

Vectorized approach is more efficient than looping.

Works for large datasets without changing code structure.

Tree-based methods improve scalability beyond brute-force.

Big-O notation describes how algorithm runtime scales with input size.

- O(N) grows linearly  
- O(N²) grows quadratically  

Better scaling does not always mean faster for small data.

In [87]:
import numpy as np

# Iris dataset (use first 2 features for visualization clarity)

X = iris_df.iloc[:, :2].to_numpy()

# 1. Compute pairwise squared distances (broadcasting)

# Shape: (N, 1, D) - (1, N, D) → (N, N, D)
diff = X[:, np.newaxis, :] - X[np.newaxis, :, :]

# Sum over feature dimension → distance matrix (N x N)
dist_sq = np.sum(diff**2, axis=-1)

print("Distance matrix shape:", dist_sq.shape)

Distance matrix shape: (150, 150)


In [88]:
# 2. Full sorting (baseline approach)

sorted_idx = np.argsort(dist_sq, axis=1)

print("\nNearest neighbors (first row):", sorted_idx[0, :5])


Nearest neighbors (first row): [ 0 17 40 43 39]


Efficient Approach: argpartition

In [89]:
k = 5

# Get indices of k+1 smallest (including self)
idx_partition = np.argpartition(dist_sq, k+1, axis=1)

# Remove self (first element may not be ordered)
neighbors_fast = idx_partition[:, 1:k+1]

print("\nFast neighbors (first row):", neighbors_fast[0])


Fast neighbors (first row): [17 40 43 39 27]


## Key Difference

| Method        | Complexity     | Output           |
|---------------|---------------|------------------|
| `argsort`     | O(N log N)    | fully ordered    |
| `argpartition`| O(N)          | partially ordered|

---

So:

k-NN does not require full sorting

Recover sorted neighbors (within K)

In [90]:
row0 = neighbors_fast[0]

sorted_local = row0[np.argsort(dist_sq[0, row0])]

print("Sorted k neighbors for first point:", sorted_local)

Sorted k neighbors for first point: [17 40 43 39 27]
